# ⬡ MediScan — Gemini Diagnostic Suite
### Technical Walkthrough Notebook

This notebook walks through the full MediScan pipeline step by step:

| Step | What Happens | Model Used |
|------|-------------|------------|
| 1 | **Initialization** — install deps, set API key, build client | — |
| 2 | **Scan Analysis** — Gemini reads the medical image/PDF and returns a structured JSON diagnosis | `gemini-3.1-flash-lite-preview` |
| 3 | **Body Map Generation** — Gemini generates an annotated anatomical illustration | `gemini-2.5-flash-image` |
| 4 | **Exercise Video Generation** — Veo generates a real instructional exercise video | `veo-3.1-lite-generate-preview` |

> ⚠️ **Note:** This notebook requires a Gemini API key with billing enabled for image + video generation.

---
## 📦 Step 1 — Install Dependencies

MediScan uses:
- `google-genai` — official Google Generative AI SDK (handles Gemini text, image, and Veo video)
- `Pillow` — for decoding and displaying generated body map images
- `PyMuPDF` — for converting PDF medical reports to images before analysis

In [1]:
!pip install google-genai Pillow PyMuPDF --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.19.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.19.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.


---
## 🔑 Step 2 — Imports & API Key Setup

We initialize the `genai.Client` with the API key. This single client object handles all three models — text, image, and video — throughout the entire pipeline.

In [2]:
import io, json, re, time
from PIL import Image
from IPython.display import display, Video
from google import genai
from google.genai import types

# ─────────────────────────────────────────────────────────────
# ▶▶  PASTE YOUR GEMINI API KEY HERE
# ─────────────────────────────────────────────────────────────
GEMINI_API_KEY = ""
# ─────────────────────────────────────────────────────────────

# Initialize the client — one client, all models
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Client initialized successfully")
print(f"   SDK version: {genai.__version__}")

✅ Client initialized successfully
   SDK version: 1.57.0


---
## 🧠 Step 3 — Scan Analysis Function

This is the core of MediScan. The function sends a medical image **or** PDF report to `gemini-3.1-flash-lite-preview` and gets back a fully structured JSON diagnosis including:

- Diagnosis text, confidence, severity, urgency
- List of affected body regions
- Clinical findings and recommendations
- A `body_map_prompt` for Step 4 (image generation)
- A list of `exercises` each with their own `illustration_prompt` for Step 5 (video generation)

**PDF vs Image handling:**
- Images → sent as `image/jpeg` bytes
- PDFs → sent as `application/pdf` directly so Gemini reads the actual text (no OCR needed)

In [3]:
def analyze_with_gemini(img_bytes: bytes, context: str = "", is_pdf: bool = False) -> dict:
    """
    Send a medical scan or PDF report to Gemini for diagnosis.
    Returns a structured JSON dict with diagnosis, findings, exercises, etc.
    
    Model: gemini-3.1-flash-lite-preview
    Input: raw bytes (JPEG image OR PDF)
    Output: dict with keys: diagnosis, severity, confidence, urgency,
            affected_regions, findings, recommendations,
            body_map_prompt, exercise_needed, exercises, disclaimer
    """
    prompt = f"""You are a senior radiologist and clinical AI. Analyze this medical {"report" if is_pdf else "image"} carefully and thoroughly.

Respond ONLY with a raw JSON object — no markdown, no code fences, no extra text.

{{
  "image_type": "X-ray | MRI | CT | Ultrasound | Report | Photo | Other",
  "diagnosis": "Primary diagnosis in clear plain language (2-3 sentences, covering ALL key findings)",
  "confidence": "high | medium | low",
  "severity": "mild | moderate | severe",
  "affected_regions": ["every specific body region affected, be detailed"],
  "findings": ["finding 1", "finding 2", "finding 3", "finding 4", "finding 5"],
  "recommendations": ["rec 1", "rec 2", "rec 3", "rec 4"],
  "body_map_prompt": "Detailed prompt for Gemini image generation: a front-view and side-view full human body medical anatomical illustration on white background, with [SPECIFIC REGIONS] highlighted in glowing red/orange with arrows and labels.",
  "exercise_needed": true,
  "exercises": [
    {{
      "name": "Exercise name",
      "purpose": "Why this specific exercise helps this specific condition",
      "difficulty": "easy | moderate | hard",
      "duration": "e.g. 10-15 minutes",
      "reps": "e.g. 3 sets of 10",
      "steps": ["Step 1", "Step 2", "Step 3"],
      "illustration_prompt": "A person performing [exercise] correctly. Clean white background, instructional style, bright studio lighting. No text overlays."
    }}
  ],
  "urgency": "routine | urgent | emergency",
  "disclaimer": "Always consult a qualified physician before acting on any AI-generated medical analysis."
}}

Give exactly 4 exercises tailored to the diagnosed condition.
{f'Patient context: {context}' if context else ''}"""

    # Build the contents list — PDF gets sent as a document, images as JPEG bytes
    if is_pdf:
        contents = [types.Part.from_bytes(data=img_bytes, mime_type="application/pdf"), prompt]
    else:
        contents = [types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"), prompt]

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=contents
    )

    # Strip any accidental markdown code fences Gemini might add
    raw = response.text.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$",     "", raw)
    raw = re.sub(r"^```\s*",     "", raw)

    return json.loads(raw)

print("✅ analyze_with_gemini() defined")

✅ analyze_with_gemini() defined


### ▶ Run the Analysis
Load a medical image or PDF and run it through Gemini. Change the file path to your own scan.

In [4]:
# ─────────────────────────────────────────────────────────────
# Point this to your medical scan file (image or PDF)
SCAN_FILE = "MRI BRAIN W_O CONTRAST.pdf"   # ← change me
CONTEXT   = "49-year-old male, headaches, history of ischemic stroke"
# ─────────────────────────────────────────────────────────────

is_pdf = SCAN_FILE.lower().endswith(".pdf")

with open(SCAN_FILE, "rb") as f:
    raw_bytes = f.read()

# For images, normalise to JPEG first
if not is_pdf:
    pil = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    buf = io.BytesIO()
    pil.save(buf, format="JPEG", quality=92)
    raw_bytes = buf.getvalue()

print(f"Loaded: {SCAN_FILE}  ({len(raw_bytes)//1024} KB, is_pdf={is_pdf})")
print("Sending to Gemini for analysis...")

result = analyze_with_gemini(raw_bytes, context=CONTEXT, is_pdf=is_pdf)

print("\n✅ Diagnosis complete!")
print(f"   Image type : {result['image_type']}")
print(f"   Severity   : {result['severity']}")
print(f"   Urgency    : {result['urgency']}")
print(f"   Confidence : {result['confidence']}")
print(f"\n📋 Diagnosis:\n   {result['diagnosis']}")
print(f"\n🔍 Findings:")
for f_ in result.get('findings', []):
    print(f"   • {f_}")
print(f"\n💊 Recommendations:")
for r_ in result.get('recommendations', []):
    print(f"   • {r_}")
print(f"\n🏋️ Exercises planned: {len(result.get('exercises', []))}")
for ex in result.get('exercises', []):
    print(f"   • {ex['name']} ({ex['difficulty']})")

Loaded: MRI BRAIN W_O CONTRAST.pdf  (107 KB, is_pdf=True)
Sending to Gemini for analysis...

✅ Diagnosis complete!
   Image type : MRI
   Severity   : moderate
   Urgency    : routine
   Confidence : high

📋 Diagnosis:
   The MRI reveals evidence of a significant prior stroke (old ischemic insult) in the right temporal and occipital regions, characterized by tissue loss and secondary wallerian degeneration. Additionally, there is evidence of generalized brain volume loss (atrophy) and small vessel ischemic changes in the white matter.

🔍 Findings:
   • Extensive tissue loss in the right temporal/occipital region
   • Ex vacuo prominence of the right lateral ventricle
   • Wallerian degeneration of the right cerebral peduncle
   • Diffuse dilatation of sulci and ventricles indicating cerebral atrophy
   • Bilateral periventricular white matter focal defects consistent with small vessel ischemic disease

💊 Recommendations:
   • Consult with a neurologist to discuss findings and manage ri

### Full JSON Output
This is the raw structured response Gemini returns — MediScan's UI reads directly from this dict.

In [5]:
print(json.dumps(result, indent=2))

{
  "image_type": "MRI",
  "diagnosis": "The MRI reveals evidence of a significant prior stroke (old ischemic insult) in the right temporal and occipital regions, characterized by tissue loss and secondary wallerian degeneration. Additionally, there is evidence of generalized brain volume loss (atrophy) and small vessel ischemic changes in the white matter.",
  "confidence": "high",
  "severity": "moderate",
  "affected_regions": [
    "Right temporal lobe",
    "Right occipital lobe",
    "Right lateral ventricle",
    "Right cerebral peduncle",
    "Bilateral periventricular white matter"
  ],
  "findings": [
    "Extensive tissue loss in the right temporal/occipital region",
    "Ex vacuo prominence of the right lateral ventricle",
    "Wallerian degeneration of the right cerebral peduncle",
    "Diffuse dilatation of sulci and ventricles indicating cerebral atrophy",
    "Bilateral periventricular white matter focal defects consistent with small vessel ischemic disease"
  ],
  "rec

---
## 🗺️ Step 4 — Body Map Image Generation

Using the `body_map_prompt` from the diagnosis result, we call `gemini-2.5-flash-image` to generate an annotated anatomical illustration highlighting exactly the affected regions Gemini identified.

**How it works:**
- We request `response_modalities=["TEXT", "IMAGE"]` to tell Gemini to return image data
- The response comes back with `inline_data` containing raw PNG/JPEG bytes
- We decode those bytes with `Pillow` and display them

In [6]:
def generate_body_map(prompt: str) -> Image.Image | None:
    """
    Generate an anatomical body map image from a text prompt.

    Model: gemini-2.5-flash-image
    Input: descriptive prompt string (generated by the diagnosis step)
    Output: PIL Image object, or None if generation failed
    """
    full_prompt = prompt + " Medical illustration, white background, anatomical precision, professional."

    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[full_prompt],
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"]  # must explicitly request IMAGE output
        ),
    )

    # The response parts can be a mix of text and image — find the image part
    for part in response.candidates[0].content.parts:
        if part.inline_data is not None:          # inline_data = raw image bytes
            return Image.open(io.BytesIO(part.inline_data.data))

    print("⚠️ No image returned by model.")
    return None

print("✅ generate_body_map() defined")

✅ generate_body_map() defined


### ▶ Generate the Body Map

In [7]:
body_map_prompt = result.get("body_map_prompt", "")
print(f"Body map prompt (from diagnosis):\n{body_map_prompt[:200]}...\n")
print("Generating body map with gemini-2.5-flash-image...")

body_map_img = generate_body_map(body_map_prompt)

if body_map_img:
    print(f"✅ Body map generated — size: {body_map_img.size}")
    body_map_img.save("body_map.png")
    print("   Saved to body_map.png")
    display(body_map_img)
else:
    print("❌ Body map generation failed.")

Body map prompt (from diagnosis):
Detailed prompt for Gemini image generation: a front-view and side-view full human body medical anatomical illustration on white background, with the brain, specifically the right temporal and occipit...

Generating body map with gemini-2.5-flash-image...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\nPlease retry in 790.007624ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api

---
## 🎬 Step 5 — Exercise Video Generation (Veo)

For each exercise in the rehabilitation plan, MediScan generates a real instructional video using `veo-3.1-lite-generate-preview`.

**How Veo works (async pattern):**
1. `client.models.generate_videos()` submits the job and returns an **operation handle** immediately
2. We poll `client.operations.get(operation)` every 10 seconds until `operation.done == True`
3. Once done, we call `client.files.download()` to fetch the raw MP4 bytes
4. The bytes are written to disk or passed to `st.video()` in the Streamlit app

> ⏱️ Each video typically takes **60–120 seconds** to generate.

In [8]:
def generate_exercise_video(prompt: str, output_filename: str) -> bool:
    """
    Generate an exercise demonstration video using Veo.

    Model: veo-3.1-lite-generate-preview
    Input: text prompt describing the exercise
    Output: saves MP4 to output_filename, returns True on success

    The API is async — we submit the job, then poll until done.
    """
    full_prompt = (
        prompt
        + " Fitness demonstration video, clear body form, bright studio lighting, "
          "plain white background, slow and instructional pace, no text overlays."
    )

    print(f"   Submitting to Veo: {prompt[:70]}...")

    # Step 1: Submit — returns immediately with an operation handle
    operation = client.models.generate_videos(
        model="veo-3.1-lite-generate-preview",
        prompt=full_prompt,
    )

    # Step 2: Poll until done (typically 60–120s)
    for attempt in range(36):          # max ~6 minutes
        time.sleep(10)
        operation = client.operations.get(operation)
        elapsed = (attempt + 1) * 10
        print(f"   Polling... {elapsed}s | done={operation.done}")
        if operation.done:
            break

    # Step 3: Download and save
    if operation.done and operation.response and operation.response.generated_videos:
        video = operation.response.generated_videos[0]
        client.files.download(file=video.video)     # populates video.video.video_bytes
        video_bytes = video.video.video_bytes
        with open(output_filename, "wb") as f:
            f.write(video_bytes)
        print(f"   ✅ Saved → {output_filename}  ({len(video_bytes)//1024} KB)")
        return True
    else:
        print(f"   ❌ No video returned. done={operation.done}")
        return False

print("✅ generate_exercise_video() defined")

✅ generate_exercise_video() defined


### ▶ Generate Exercise Videos
We loop through all 4 exercises from the diagnosis result and generate a video for each.

In [9]:
exercises = result.get("exercises", [])
print(f"Generating videos for {len(exercises)} exercises...\n")

generated_videos = []

for i, ex in enumerate(exercises[:4]):
    name     = ex.get("name", f"exercise_{i+1}")
    ex_prompt = ex.get("illustration_prompt", f"A person performing {name} correctly.")
    filename  = f"exercise_{i+1}_{name.lower().replace(' ', '_')[:30]}.mp4"

    print(f"\n[{i+1}/{len(exercises[:4])}] {name}")
    success = generate_exercise_video(ex_prompt, filename)
    if success:
        generated_videos.append(filename)

print(f"\n✅ Done — {len(generated_videos)}/{len(exercises[:4])} videos generated")

Generating videos for 4 exercises...


[1/4] Brisk Walking
   Submitting to Veo: A middle-aged man walking briskly in a park on a sunny day. Clean whit...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

### ▶ Preview the Videos

In [10]:
for filename in generated_videos:
    print(f"▶ {filename}")
    display(Video(filename, embed=True, width=640))

---
## 🏗️ How It All Connects — The Full Pipeline

```
Medical Scan / PDF
        │
        ▼
┌─────────────────────────────────────┐
│  gemini-3.1-flash-lite-preview      │  ← reads image or PDF text
│  analyze_with_gemini()              │  ← returns structured JSON
└───────────┬─────────────────────────┘
            │
     ┌──────┴──────┐
     │             │
     ▼             ▼
body_map_prompt   exercises[]
     │             │
     ▼             ▼
┌──────────┐  ┌────────────────────────┐
│ gemini   │  │ veo-3.1-lite-generate  │
│ 2.5-flash│  │ -preview               │
│ -image   │  │ (one job per exercise) │
└────┬─────┘  └──────────┬─────────────┘
     │                   │
     ▼                   ▼
 body_map.png      exercise_N.mp4 x4
     │                   │
     └─────────┬─────────┘
               ▼
       Streamlit UI renders
       diagnosis + map + videos
```

In the Streamlit app (`mediscan_gemini.py`), all of this runs inside the `if analyze_btn:` block, with `st.session_state` caching the results so the UI doesn't re-run on every interaction.

---
## ✅ Summary

| Component | Function | Model | Output |
|-----------|----------|-------|--------|
| Diagnosis | `analyze_with_gemini()` | `gemini-3.1-flash-lite-preview` | JSON dict |
| Body Map | `generate_body_map()` | `gemini-2.5-flash-image` | PIL Image / PNG |
| Exercise Videos | `generate_exercise_video()` | `veo-3.1-lite-generate-preview` | MP4 bytes |

The Streamlit app wraps all three functions with a UI for file upload, spinners during generation, and styled cards for displaying results.

> 💡 All three models share the same `genai.Client` instance — one API key, one client, everything.